In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import os
base = '/content/drive/MyDrive/phishing-project'
os.makedirs(base + '/data', exist_ok=True)
print(os.listdir(base))

['data']


In [11]:
import pandas as pd
import requests

url = "http://data.phishtank.com/data/online-valid.csv"
headers = {"User-Agent": "phishtank/your-college-project"}

r = requests.get(url, headers=headers, timeout=120)
print("Status:", r.status_code)

if r.status_code == 200:
    path = base + "/data/phishtank_raw.csv"
    with open(path, "wb") as f:
        f.write(r.content)
    df = pd.read_csv(path)
    print(df.shape)
    print(df.columns.tolist())
    print(df[["url", "submission_time"]].head())

Status: 200
(53171, 8)
['phish_id', 'url', 'phish_detail_url', 'submission_time', 'verified', 'verification_time', 'online', 'target']
                                              url            submission_time
0  https://poonkiewaters.com/ipko/pages/login.php  2026-10-30T15:52:22+00:00
1                        http://tf-app.vercel.app  2026-10-30T15:25:13+00:00
2                      https://tf-app.vercel.app/  2026-10-30T15:25:13+00:00
3     https://faciliteagora.vercel.app/index.html  2026-10-30T14:17:20+00:00
4             https://farm-rio-outlet.vercel.app/  2026-10-30T13:28:17+00:00


In [12]:
df["submission_time"] = pd.to_datetime(df["submission_time"])

print("Earliest:", df["submission_time"].min())
print("Latest:  ", df["submission_time"].max())
print("Unique URLs:", df["url"].nunique(), "/", len(df))

print(df["submission_time"].dt.to_period("M").value_counts().sort_index().tail(24))

Earliest: 2011-02-18 10:50:32+00:00
Latest:   2026-10-30 15:52:22+00:00
Unique URLs: 53169 / 53171
submission_time
2024-11    1452
2024-12     896
2025-01     599
2025-02     424
2025-03     526
2025-04     537
2025-05     548
2025-06     405
2025-07     926
2025-08     996
2025-09     882
2025-10     799
2025-11     729
2025-12     470
2026-01    2278
2026-02    3468
2026-03     939
2026-04    1222
2026-05    2109
2026-06    2414
2026-07    2614
2026-08    2130
2026-09    1380
2026-10    2479
Freq: M, Name: count, dtype: int64


/tmp/ipykernel_3200/1751158847.py:7: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  print(df["submission_time"].dt.to_period("M").value_counts().sort_index().tail(24))


In [13]:
phish = df[["url", "submission_time"]].copy()
phish["submission_time"] = phish["submission_time"].dt.tz_convert(None)

phish = phish[phish["submission_time"] >= "2024-11-01"]
phish = phish.drop_duplicates("url")
phish["label"] = 1
phish["source"] = "phishtank"

phish.to_csv(base + "/data/phish_clean.csv", index=False)
print(phish.shape)
print(phish["submission_time"].min(), "to", phish["submission_time"].max())
phish.head()

(31222, 4)
2024-11-01 06:35:59 to 2026-10-30 15:52:22


,url,submission_time,label,source
0,https://poonkiewaters.com/ipko/pages/login.php,2026-10-30 15:52:22,1,phishtank
1,http://tf-app.vercel.app,2026-10-30 15:25:13,1,phishtank
2,https://tf-app.vercel.app/,2026-10-30 15:25:13,1,phishtank
3,https://faciliteagora.vercel.app/index.html,2026-10-30 14:17:20,1,phishtank
4,https://farm-rio-outlet.vercel.app/,2026-10-30 13:28:17,1,phishtank


In [14]:
import zipfile, io

r = requests.get("https://tranco-list.eu/top-1m.csv.zip", timeout=120)
print("Status:", r.status_code)

z = zipfile.ZipFile(io.BytesIO(r.content))
tranco = pd.read_csv(z.open("top-1m.csv"), names=["rank", "domain"])

tranco.to_csv(base + "/data/tranco_raw.csv", index=False)
print(tranco.shape)
tranco.head(10)

Status: 200
(1000000, 2)


,rank,domain
0,1,google.com
1,2,cloudflare.com
2,3,amazonaws.com
3,4,ezviz7.com
4,5,facebook.com
5,6,fbcdn.net
6,7,gstatic.com
7,8,googleapis.com
8,9,akamai.net
9,10,microsoft.com


In [18]:
import json, time, random, requests
import pandas as pd

# 1. Server check
try:
    info = requests.get("https://index.commoncrawl.org/collinfo.json", timeout=30).json()
    cdx_api = info[0]["cdx-api"]
    print("Server OK, using:", info[0]["id"])
    server_ok = True
except Exception as e:
    print("Server abhi bhi nahi chal raha:", type(e).__name__)
    server_ok = False

# 2. Pilot (sirf server OK ho to)
if server_ok:
    def cc_urls(domain, limit=10, retries=2):
        params = {
            "url": domain,
            "matchType": "domain",
            "output": "json",
            "fl": "url",
            "filter": ["=status:200", "=mime:text/html"],
            "collapse": "urlkey",
            "limit": limit,
        }
        for attempt in range(retries + 1):
            try:
                r = requests.get(cdx_api, params=params, timeout=60)
                if r.status_code == 200:
                    text = r.text.strip()
                    return [json.loads(l)["url"] for l in text.split("\n") if l]
            except requests.exceptions.RequestException:
                pass
            time.sleep(3)
        return None

    random.seed(42)
    pool = tranco[tranco["rank"] <= 20000]["domain"].tolist()
    sample = random.sample(pool, 60)

    rows, failed, streak = [], [], 0
    for i, d in enumerate(sample):
        urls = cc_urls(d)
        if urls is None:
            failed.append(d)
            streak += 1
            if streak >= 5:
                print("Lagatar 5 domains fail hue, ruk raha hoon.")
                break
        else:
            streak = 0
            rows += [(u, d) for u in urls]
        time.sleep(1)
        if (i + 1) % 10 == 0:
            print(f"{i+1}/60 done")

    legit = pd.DataFrame(rows, columns=["url", "domain"])
    legit.to_csv(base + "/data/legit_pilot.csv", index=False)

    print("Domains failed:", len(failed), "/ 60")
    print("Total URLs:", len(legit))
    print("With paths:", (legit["url"].str.count("/") > 3).sum())
    print(legit.sample(min(10, len(legit)), random_state=1)["url"].to_string())

Server OK, using: CC-MAIN-2026-43
Lagatar 5 domains fail hue, ruk raha hoon.
Domains failed: 5 / 60
Total URLs: 0
With paths: 0
Series([], )


In [19]:
import requests, json, time

def test(label, api, params):
    try:
        r = requests.get(api, params=params, timeout=60)
        print(f"{label} -> HTTP {r.status_code} | {r.text[:120]!r}")
    except Exception as e:
        print(f"{label} -> {type(e).__name__}")

# Wo domains jo pehle chale the (wikipedia, github) + ek random
for d in ["wikipedia.org", "github.com", "bbc.com"]:
    # Light query: domain/* , bina matchType ke
    test(f"{d} (light, latest crawl)", cdx_api,
         {"url": f"{d}/*", "output": "json", "fl": "url", "limit": 5})
    time.sleep(2)

# Ek purana crawl index try karo
old_api = info[1]["cdx-api"]
print("Older crawl:", info[1]["id"])
test("wikipedia.org (older crawl)", old_api,
     {"url": "wikipedia.org/*", "output": "json", "fl": "url", "limit": 5})

wikipedia.org (light, latest crawl) -> ConnectionError
github.com (light, latest crawl) -> ConnectionError
bbc.com (light, latest crawl) -> ConnectionError
Older crawl: CC-MAIN-2026-39
wikipedia.org (older crawl) -> ConnectionError


In [20]:
import requests, random
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

HEADERS = {"User-Agent": "Mozilla/5.0 (college research project)"}
SKIP_EXT = (".jpg", ".jpeg", ".png", ".gif", ".svg", ".css", ".js", ".pdf",
            ".zip", ".ico", ".mp4", ".xml", ".woff", ".woff2")

def get_links(domain, per_domain=10):
    try:
        r = requests.get(f"https://{domain}", headers=HEADERS, timeout=10)
        if r.status_code != 200 or "text/html" not in r.headers.get("Content-Type", ""):
            return []
        soup = BeautifulSoup(r.text, "html.parser")
        out = set()
        for a in soup.find_all("a", href=True):
            u = urljoin(r.url, a["href"]).split("#")[0]
            p = urlparse(u)
            if p.scheme not in ("http", "https"):
                continue
            host = p.netloc.lower()
            if not (host == domain or host.endswith("." + domain)):
                continue
            if len(p.path) <= 1 and not p.query:
                continue
            if p.path.lower().endswith(SKIP_EXT):
                continue
            out.add(u)
        out = list(out)
        random.shuffle(out)
        return out[:per_domain]
    except Exception:
        return []

random.seed(42)
pool = tranco[tranco["rank"] <= 20000]["domain"].tolist()
sample = random.sample(pool, 60)

with ThreadPoolExecutor(max_workers=20) as ex:
    results = list(ex.map(get_links, sample))

rows = [(u, d) for d, urls in zip(sample, results) for u in urls]
legit = pd.DataFrame(rows, columns=["url", "domain"])
legit.to_csv(base + "/data/legit_pilot.csv", index=False)

print("Domains that gave links:", sum(1 for r in results if r), "/ 60")
print("Total URLs:", len(legit))
print(legit.sample(min(10, len(legit)), random_state=1)["url"].to_string())

Domains that gave links: 32 / 60
Total URLs: 300
189    https://www.aok.de/pk/news/grippeimpfung-wer-s...
123                https://dstillery.com/audience-studio
185          https://www.aok.de/pk/magazin/wohlbefinden/
213    https://www.ccc.de/club/chaostreffs/chaostreff...
106    https://www.nordea.com/en/about-us/corporate-g...
127          https://media.buyee.jp/pr/about_auction/en/
176                                https://maps.me/maps/
73                      https://material.security/events
275          https://avtub.wiki/category/jav-uncensored/
242            https://www.socialmediatoday.com/contact/


In [22]:
"""
Legitimate URL collection.

Samples domains from the Tranco top-100k list, fetches each homepage, and
extracts same-site links that contain a path or query string. Results are
saved in chunks to Google Drive so the job can resume after a disconnect.
"""

import os
import random
import time
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

# Configuration
HEADERS = {"User-Agent": "Mozilla/5.0 (academic research project)"}
SKIP_EXTENSIONS = (".jpg", ".jpeg", ".png", ".gif", ".svg", ".css", ".js", ".pdf",
                   ".zip", ".ico", ".mp4", ".xml", ".woff", ".woff2")
URLS_PER_DOMAIN = 10
N_DOMAINS = 6000
CHUNK_SIZE = 500
MAX_WORKERS = 40
SEED = 7


def get_links(domain, per_domain=URLS_PER_DOMAIN):
    """Return up to `per_domain` same-site URLs (with a path or query) from a domain's homepage."""
    try:
        resp = requests.get(f"https://{domain}", headers=HEADERS, timeout=10)
        if resp.status_code != 200 or "text/html" not in resp.headers.get("Content-Type", ""):
            return []

        soup = BeautifulSoup(resp.text, "html.parser")
        links = set()
        for anchor in soup.find_all("a", href=True):
            url = urljoin(resp.url, anchor["href"]).split("#")[0]
            parsed = urlparse(url)

            if parsed.scheme not in ("http", "https"):
                continue
            host = parsed.netloc.lower()
            if not (host == domain or host.endswith("." + domain)):
                continue  # keep same-site links only
            if len(parsed.path) <= 1 and not parsed.query:
                continue  # skip bare homepages
            if parsed.path.lower().endswith(SKIP_EXTENSIONS):
                continue  # skip static assets

            links.add(url)

        links = list(links)
        random.shuffle(links)
        return links[:per_domain]
    except Exception:
        return []  # unreachable, blocked, or malformed response


# Domain sampling
random.seed(SEED)
pool = tranco[tranco["rank"] <= 100_000]
sample = pool.sample(N_DOMAINS, random_state=SEED)
domains = sample["domain"].tolist()
rank_of = dict(zip(sample["domain"], sample["rank"]))

chunk_dir = os.path.join(base, "data", "legit_chunks")
os.makedirs(chunk_dir, exist_ok=True)

# Chunked collection with checkpointing
n_chunks = len(domains) // CHUNK_SIZE
start_time = time.time()

for chunk_idx in range(n_chunks):
    chunk_path = os.path.join(chunk_dir, f"chunk_{chunk_idx:02d}.csv")
    if os.path.exists(chunk_path):
        continue  # checkpoint exists, skip this chunk

    batch = domains[chunk_idx * CHUNK_SIZE:(chunk_idx + 1) * CHUNK_SIZE]
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        results = list(executor.map(get_links, batch))

    rows = [(url, dom, rank_of[dom]) for dom, urls in zip(batch, results) for url in urls]
    pd.DataFrame(rows, columns=["url", "domain", "rank"]).to_csv(chunk_path, index=False)

    elapsed = int(time.time() - start_time)
    print(f"Chunk {chunk_idx + 1}/{n_chunks} complete | {len(rows)} URLs | {elapsed}s elapsed")

print("Collection complete.")

Chunk 4/12 complete | 2125 URLs | 45s elapsed
Chunk 5/12 complete | 2139 URLs | 148s elapsed
Chunk 6/12 complete | 2250 URLs | 183s elapsed
Chunk 7/12 complete | 2207 URLs | 216s elapsed
Chunk 8/12 complete | 1915 URLs | 248s elapsed
Chunk 9/12 complete | 2225 URLs | 467s elapsed
Chunk 10/12 complete | 2346 URLs | 511s elapsed
Chunk 11/12 complete | 2202 URLs | 630s elapsed
Chunk 12/12 complete | 2018 URLs | 666s elapsed
Collection complete.


In [23]:
!pip install -q tldextract

import glob
import pandas as pd
import tldextract

# Load legitimate URL chunks and phishing data
chunk_files = sorted(glob.glob(base + "/data/legit_chunks/chunk_*.csv"))
print("Legitimate chunk files found:", len(chunk_files))

legit = pd.concat([pd.read_csv(f) for f in chunk_files], ignore_index=True)
phish = pd.read_csv(base + "/data/phish_clean.csv", parse_dates=["submission_time"])
print(f"Raw counts -> legitimate: {len(legit)}, phishing: {len(phish)}")

# Clean legitimate data
legit["url"] = legit["url"].str.strip()
legit = legit.drop_duplicates("url").reset_index(drop=True)
legit["label"] = 0
legit["source"] = "tranco_homepage_links"
legit["submission_time"] = pd.NaT  # no reliable timestamp for legitimate URLs

# Clean phishing data
phish["url"] = phish["url"].str.strip()
phish = phish.drop_duplicates("url").reset_index(drop=True)
phish["rank"] = pd.NA

# Combine
cols = ["url", "label", "source", "submission_time", "rank"]
df_all = pd.concat([phish[cols], legit[cols]], ignore_index=True)
df_all = df_all.drop_duplicates("url").reset_index(drop=True)

# Registered domain (needed later for domain-disjoint splits)
def registered_domain(url):
    ext = tldextract.extract(url)
    return f"{ext.domain}.{ext.suffix}" if ext.suffix else ext.domain

df_all["domain"] = df_all["url"].map(registered_domain)

# Save
df_all.to_csv(base + "/data/dataset_v1.csv", index=False)

# Summary
print("\nClass counts:")
print(df_all["label"].value_counts())
print("\nUnique registered domains per class:")
print(df_all.groupby("label")["domain"].nunique())

overlap = set(df_all[df_all.label == 1]["domain"]) & set(df_all[df_all.label == 0]["domain"])
print("\nDomains present in both classes:", len(overlap))
print(sorted(overlap)[:15])

df_all["url_len"] = df_all["url"].str.len()
df_all["n_slashes"] = df_all["url"].str.count("/")
print("\nURL length and slash count by class:")
print(df_all.groupby("label")[["url_len", "n_slashes"]].describe().T.loc[
    [("url_len", "mean"), ("url_len", "50%"), ("n_slashes", "mean"), ("n_slashes", "50%")]
])

Legitimate chunk files found: 12
Raw counts -> legitimate: 25938, phishing: 31222

Class counts:
label
1    31222
0    25937
Name: count, dtype: int64

Unique registered domains per class:
label
0    2788
1    7963
Name: domain, dtype: int64

Domains present in both classes: 7
['benefitsaccess-center.com', 'framer.com', 'google.com.hk', 'link.me', 'ramp.com', 'tr.ee', 'yam.com']

URL length and slash count by class:
label                   0          1
url_len   mean  54.618229  50.958235
          50%   46.000000  37.000000
n_slashes mean   4.292632   3.408686
          50%    4.000000   3.000000


In [24]:
import pandas as pd

df_all = pd.read_csv(base + "/data/dataset_v1.csv", parse_dates=["submission_time"])

# Domains that appear in both classes (e.g. free hosting or URL shorteners)
dom_by_class = df_all.groupby("domain")["label"].nunique()
overlap = set(dom_by_class[dom_by_class > 1].index)
print("Overlapping domains removed:", sorted(overlap))

df_final = df_all[~df_all["domain"].isin(overlap)].reset_index(drop=True)
df_final.to_csv(base + "/data/dataset_final_v1.csv", index=False)

print("\nFinal shape:", df_final.shape)
print(df_final["label"].value_counts())
print("\nUnique domains per class:")
print(df_final.groupby("label")["domain"].nunique())
print("\nPhishing date range:",
      df_final[df_final.label == 1]["submission_time"].min(), "to",
      df_final[df_final.label == 1]["submission_time"].max())


Overlapping domains removed: ['benefitsaccess-center.com', 'framer.com', 'google.com.hk', 'link.me', 'ramp.com', 'tr.ee', 'yam.com']

Final shape: (57052, 6)
label
1    31178
0    25874
Name: count, dtype: int64

Unique domains per class:
label
0    2781
1    7956
Name: domain, dtype: int64

Phishing date range: 2024-11-01 06:35:59 to 2026-10-30 15:52:22
